# Community Detection

### Import libraries

In [1]:
import math
import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns
import time
import dimod
from networkx.algorithms.community.quality import modularity as nx_modularity
from dwave.system import LeapHybridSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.system import LeapHybridDQMSampler
from dwave.samplers import SimulatedAnnealingSampler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

### Load data

In [93]:
graphs = {}

for i in range(5, 20):
    G = nx.read_graphml(f"neuronal_graphs/neurons{i}.graphml")
    G.remove_nodes_from(list(nx.isolates(G)))
    print(f"Graph {i-4} loaded for r > {i*0.01}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    graphs[i] = {'G': G}

print(graphs)

Graph 1 loaded for r > 0.05: 124 nodes, 246 edges
Graph 2 loaded for r > 0.06: 114 nodes, 196 edges
Graph 3 loaded for r > 0.07: 107 nodes, 164 edges
Graph 4 loaded for r > 0.08: 100 nodes, 140 edges
Graph 5 loaded for r > 0.09: 89 nodes, 118 edges
Graph 6 loaded for r > 0.1: 77 nodes, 96 edges
Graph 7 loaded for r > 0.11: 68 nodes, 82 edges
Graph 8 loaded for r > 0.12: 61 nodes, 70 edges
Graph 9 loaded for r > 0.13: 55 nodes, 62 edges
Graph 10 loaded for r > 0.14: 48 nodes, 52 edges
Graph 11 loaded for r > 0.15: 42 nodes, 46 edges
Graph 12 loaded for r > 0.16: 38 nodes, 42 edges
Graph 13 loaded for r > 0.17: 27 nodes, 30 edges
Graph 14 loaded for r > 0.18: 26 nodes, 28 edges
Graph 15 loaded for r > 0.19: 25 nodes, 26 edges
{5: {'G': <networkx.classes.digraph.DiGraph object at 0x0000022B62F30E90>}, 6: {'G': <networkx.classes.digraph.DiGraph object at 0x0000022B63170290>}, 7: {'G': <networkx.classes.digraph.DiGraph object at 0x0000022B63175790>}, 8: {'G': <networkx.classes.digraph.DiGra

# 1 Graph preprocessing

### 1.1 Conversion to adjacency matrix

In [94]:
for i in range(5, 20):
    G = graphs[i]['G']
    # Convert directed graph to undirected
    G_undirected = G.to_undirected()

    # Remove any remaining self-loops
    G_undirected.remove_edges_from(nx.selfloop_edges(G_undirected))

    # Create unweighted adjacency matrix
    A = nx.adjacency_matrix(G_undirected, weight=None).toarray()
    N = G_undirected.number_of_nodes()
    M = G_undirected.number_of_edges()  # Total number of edges

    graphs[i]['G_undirected'] = G_undirected
    graphs[i]['A'] = A
    graphs[i]['N'] = N
    graphs[i]['M'] = M

    print(f"\nGraph {i-4} summary:")
    print(f"\nUndirected graph: {N} nodes, {M} edges")
    print(f"Adjacency matrix shape: {A.shape}")
    print(f"Sparsity: {(A == 0).sum() / A.size * 100:.2f}% zeros")
    print(f"\nDegree statistics:")
    degrees = [G_undirected.degree(n) for n in G_undirected.nodes()]
    print(f"  Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.2f}")


Graph 1 summary:

Undirected graph: 124 nodes, 123 edges
Adjacency matrix shape: (124, 124)
Sparsity: 98.40% zeros

Degree statistics:
  Min: 1, Max: 6, Mean: 1.98

Graph 2 summary:

Undirected graph: 114 nodes, 98 edges
Adjacency matrix shape: (114, 114)
Sparsity: 98.49% zeros

Degree statistics:
  Min: 1, Max: 5, Mean: 1.72

Graph 3 summary:

Undirected graph: 107 nodes, 82 edges
Adjacency matrix shape: (107, 107)
Sparsity: 98.57% zeros

Degree statistics:
  Min: 1, Max: 4, Mean: 1.53

Graph 4 summary:

Undirected graph: 100 nodes, 70 edges
Adjacency matrix shape: (100, 100)
Sparsity: 98.60% zeros

Degree statistics:
  Min: 1, Max: 3, Mean: 1.40

Graph 5 summary:

Undirected graph: 89 nodes, 59 edges
Adjacency matrix shape: (89, 89)
Sparsity: 98.51% zeros

Degree statistics:
  Min: 1, Max: 3, Mean: 1.33

Graph 6 summary:

Undirected graph: 77 nodes, 48 edges
Adjacency matrix shape: (77, 77)
Sparsity: 98.38% zeros

Degree statistics:
  Min: 1, Max: 3, Mean: 1.25

Graph 7 summary:

Un

# 2 Classical community detection

In [95]:
def communities_to_partition(communities):
    """
    Convert a list of community sets to a partition array.
    Returns: array where partition[i] = community_id for node i
    """
    node_list = list(G_undirected.nodes())  # Use the current graph's node order
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    partition = np.zeros(len(node_list), dtype=int)
    for community_id, community_set in enumerate(communities):
        for node in community_set:
            partition[node_to_idx[node]] = community_id
    return partition

def compute_modularity(partition, A):
    """
    Compute modularity Q for a given partition.
    Q = (1/2m) * sum_ij (A_ij - k_i*k_j/2m) * delta(c_i, c_j)
    where k_i is the degree of node i, and delta(c_i, c_j) = 1 if nodes i,j in same community
    """
    n = A.shape[0]

    degrees = A.sum(axis=1)
    m = A.sum() / 2.0  # total edges (double counted in adjacency matrix)
    
    Q = 0.0
    for i in range(n):
        for j in range(n):
            if partition[i] == partition[j]:
                Q += A[i, j] - (degrees[i] * degrees[j]) / (2.0 * m)
    
    Q /= (2.0 * m)
    return Q

def compute_conductance(partition, A):
    """
    Compute conductance for a given partition.
    Conductance = cut(S, S') / min(vol(S), vol(S'))
    where cut(S, S') is the number of edges between S and its complement,
    and vol(S) is the sum of degrees in S.
    """
    degrees = A.sum(axis=1)
    conductances = []
    for community_id in np.unique(partition):
        S = np.where(partition == community_id)[0]

        # Skip single-node communities (undefined conductance)
        if len(S) <= 1:
            continue

        S_complement = np.where(partition != community_id)[0]
        cut = A[np.ix_(S, S_complement)].sum()
        vol_S = degrees[S].sum()
        vol_S_complement = degrees[S_complement].sum()

        if min(vol_S, vol_S_complement) > 0:
            conductance = cut / min(vol_S, vol_S_complement)
            conductances.append(conductance)

    return np.mean(conductances) if conductances else 0.0

### 2.1 Louvain

In [104]:
louvain_stats = pd.DataFrame(columns=['threshold', 'partition', 'runtime', 'num_communities', 'modularity', 'conductance', 'community_sizes'])

for i in range(5, 20):
    G_undirected = graphs[i]['G_undirected']
    A = graphs[i]['A']
    start_time = time.time()
    communities_louvain = list(nx.community.louvain_communities(G_undirected, seed=42))
    louvain_runtime = time.time() - start_time

    partition_louvain = communities_to_partition(communities_louvain)
    modularity_louvain = compute_modularity(partition_louvain, A)
    conductance_louvain = compute_conductance(partition_louvain, A)

    louvain_stats.loc[len(louvain_stats)] = {
        'threshold': i*0.01,
        'partition': partition_louvain,
        'runtime': louvain_runtime,
        'num_communities': len(communities_louvain),
        'modularity': modularity_louvain,
        'conductance': conductance_louvain,
        'community_sizes': sorted([len(c) for c in communities_louvain], reverse=True)
    }

    print(f"\nGraph {i-4} results:")
    print(f"Runtime: {louvain_runtime:.4f}s")
    print(f"Number of communities: {len(communities_louvain)}")
    print(f"Modularity: {modularity_louvain:.6f}")
    print(f"Conductance: {conductance_louvain:.6f}")
    print(f"Community sizes: {sorted([len(c) for c in communities_louvain], reverse=True)}")
    print(f"Number of communities: {len(communities_louvain)}")

louvain_stats.to_csv('stats_by_threshold/louvain_stats.csv', index=False)


Graph 1 results:
Runtime: 0.0208s
Number of communities: 55
Modularity: 0.530240
Conductance: 0.285524
Community sizes: [11, 8, 8, 7, 6, 5, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of communities: 55

Graph 2 results:
Runtime: 0.0107s
Number of communities: 57
Modularity: 0.568149
Conductance: 0.246675
Community sizes: [7, 7, 6, 6, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of communities: 57

Graph 3 results:
Runtime: 0.0067s
Number of communities: 61
Modularity: 0.544244
Conductance: 0.250312
Community sizes: [5, 5, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of communities: 61

Graph 4 results:
Runtime: 0.0046s
N

### 2.2 Greedy w/ best_n

In [105]:
greedy_stats = pd.DataFrame(columns=['threshold', 'partition', 'runtime', 'num_communities', 'modularity', 'conductance', 'community_sizes'])

for i in range(5, 20):
    G_undirected = graphs[i]['G_undirected']
    A = graphs[i]['A']
    N = graphs[i]['N']
    start_time = time.time()
    communities_greedy = list(nx.community.greedy_modularity_communities(G_undirected, best_n=math.sqrt(N)))
    greedy_runtime = time.time() - start_time

    partition_greedy = communities_to_partition(communities_greedy)
    modularity_greedy = compute_modularity(partition_greedy, A)
    conductance_greedy = compute_conductance(partition_greedy, A)

    greedy_stats.loc[len(greedy_stats)] = {
        'threshold': i*0.01,
        'partition': partition_greedy,
        'runtime': greedy_runtime,
        'num_communities': len(communities_greedy),
        'modularity': modularity_greedy,
        'conductance': conductance_greedy,
        'community_sizes': sorted([len(c) for c in communities_greedy], reverse=True)
    }

    print(f"\nGraph {i-4} results:")
    print(f"Runtime: {greedy_runtime:.4f}s")
    print(f"Number of communities: {len(communities_greedy)}")
    print(f"Modularity: {modularity_greedy:.6f}")
    print(f"Conductance: {conductance_greedy:.6f}")
    print(f"Community sizes: {sorted([len(c) for c in communities_greedy], reverse=True)}")
    print(f"Number of communities: {len(communities_greedy)}")

greedy_stats.to_csv('stats_by_threshold/greedy_stats.csv', index=False)


Graph 1 results:
Runtime: 0.0139s
Number of communities: 11
Modularity: 0.254743
Conductance: 0.000000
Community sizes: [97, 5, 4, 3, 3, 2, 2, 2, 2, 2, 2]
Number of communities: 11

Graph 2 results:
Runtime: 0.0066s
Number of communities: 10
Modularity: 0.174302
Conductance: 0.000000
Community sizes: [96, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Number of communities: 10

Graph 3 results:
Runtime: 0.0040s
Number of communities: 10
Modularity: 0.206127
Conductance: 0.000000
Community sizes: [89, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Number of communities: 10

Graph 4 results:
Runtime: 0.0031s
Number of communities: 10
Modularity: 0.238776
Conductance: 0.000000
Community sizes: [82, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Number of communities: 10

Graph 5 results:
Runtime: 0.0035s
Number of communities: 9
Modularity: 0.250503
Conductance: 0.000000
Community sizes: [73, 2, 2, 2, 2, 2, 2, 2, 2]
Number of communities: 9

Graph 6 results:
Runtime: 0.0017s
Number of communities: 8
Modularity: 0.267361
Conductance: 0.000000
Comm

# 3 QUBO with LeapHybrd

### Define functions

In [106]:
def build_modularity_qubo(A, num_communities=None, lambda_penalty=None, verbose=False):
    """
    Build QUBO matrix for modularity maximization via binary community assignment.
    Variables: x_{i,c} for i in nodes, c in communities
    Constraint: Each node assigned to exactly one community (handled via penalty)
    
    One-hot encoding:
    - num_binary_vars = N * num_communities
    - x[i*num_communities + c] = 1 if node i in community c
    
    QUBO objective: minimize -Q + lambda * penalty
    
    Parameters:
    -----------
    A : ndarray
        Adjacency matrix
    num_communities : int, optional
        Number of communities (default: sqrt(N))
    lambda_penalty : float, optional
        Penalty weight for one-hot constraint.
        If None, automatically set to 10x the max modularity contribution.
    verbose : bool
        Print diagnostics
    """

    if num_communities is None:
        num_communities = max(2, int(np.sqrt(N)))

    n_vars = N * num_communities 
    Q = {}

    degrees = A.sum(axis=1)
    m = A.sum() / 2  # Total edges

    # Modularity term: Q = (1/2m) * sum_{c} sum_{i,j} B_{ij} x_{i,c} x_{j,c}
    # For minimization, use -Q in QUBO objective
    max_B = 0  # Track max modularity contribution for scaling
    
    for c in range(num_communities):
        for i in range(N):
            for j in range(i, N):
                var_i = i * num_communities + c
                var_j = j * num_communities + c
                B_ij = A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
                coeff = -B_ij / (2 * m)
                max_B = max(max_B, abs(coeff))

                if i == j:  # diagonal
                    if var_i not in Q:
                        Q[var_i] = 0.0
                    Q[var_i] += coeff
                else:  # off-diagonal
                    key = (var_i, var_j) if var_i < var_j else (var_j, var_i)
                    if key not in Q:
                        Q[key] = 0.0
                    Q[key] += coeff

    if lambda_penalty is None:
        lambda_penalty = max(10.0 * max_B, 5.0)  # At least 5.0
    
    if verbose:
        print(f"Modularity scale (max |coeff|): {max_B:.6f}")
        print(f"Penalty weight (lambda): {lambda_penalty:.6f}")

    # Constraint penalty: each node in exactly one community
    # Penalty: lambda * (sum_c x_{i,c} - 1)^2
    # Expansion:
    #   (sum_c x_{i,c} - 1)^2 = sum_c x_{i,c}^2 + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1
    #   = sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} - 2*sum_c x_{i,c} + 1   [since x^2=x for binary]
    #   = -sum_c x_{i,c} + 2*sum_{c1<c2} x_{i,c1}*x_{i,c2} + 1
    
    # Linear coefficient: -lambda per variable
    # Quadratic coefficient: +2*lambda per pair

    const_term = N * lambda_penalty
    
    for i in range(N):
        # Linear terms: -lambda * x_{i,c}
        for c in range(num_communities):
            var_i = i * num_communities + c
            if var_i not in Q:
                Q[var_i] = 0.0
            Q[var_i] += -lambda_penalty

        # Quadratic terms: +2*lambda * x_{i,c1}*x_{i,c2}
        for c1 in range(num_communities):
            for c2 in range(c1 + 1, num_communities):
                var_1 = i * num_communities + c1
                var_2 = i * num_communities + c2
                key = (var_1, var_2) if var_1 < var_2 else (var_2, var_1)
                if key not in Q:
                    Q[key] = 0.0
                Q[key] += 2.0 * lambda_penalty

    Q['__const__'] = const_term
    
    if verbose:
        print(f"Constant term (N*lambda): {const_term:.2f}")
        print(f"Total QUBO terms: {len(Q)-1} + 1 const")  # -1 for const term itself

    return Q, num_communities

def extract_communities_from_sample(sample, N, k_communities):
    """
    Extract community assignments from QUBO sample.
    For node i, find c where x_{i,c} = 1
    """
    partition = np.zeros(N, dtype=int)
    
    for i in range(N):
        # Find which community this node is assigned to
        for c in range(k_communities):
            var_idx = i * k_communities + c
            if var_idx in sample and sample[var_idx] == 1:
                partition[i] = c
                break
        else:
            # If no community assigned (constraint violation), assign to community 0
            partition[i] = 0
    
    return partition
count = 1

### Define refinement functions

In [107]:
def greedy_refinement(A, partition):
    """
    Refines the partition using a greedy local search.
    Moves each node to the community that gives the highest modularity increase.
    """
    refined_partition = partition.copy()
    N = A.shape[0]
    unique_communities = np.unique(partition)
    improved = True
    
    current_modularity = compute_modularity(refined_partition, A)
    
    print(f"Starting refinement... Initial Modularity: {current_modularity:.6f}")
    
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        
        for i in nodes:
            best_move = refined_partition[i]
            initial_comm = refined_partition[i]
            for target_comm in unique_communities:
                if target_comm == initial_comm:
                    continue
                refined_partition[i] = target_comm
                new_modularity = compute_modularity(refined_partition, A)
                if new_modularity > current_modularity:
                    current_modularity = new_modularity
                    best_move = target_comm
                    improved = True
            refined_partition[i] = best_move
            
    print(f"Refinement complete. Optimized Modularity: {current_modularity:.6f}")
    return refined_partition

def advanced_refinement(A, partition):
    N = A.shape[0]
    m = np.sum(A) / 2
    degrees = np.sum(A, axis=1)
    refined_partition = partition.copy()
    
    def get_delta_q(node_i, target_comm, curr_partition):
        # Calculate only the change in modularity for moving node_i
        # Internal edges to the target community
        comm_nodes = np.where(curr_partition == target_comm)[0]
        ki_in = np.sum(A[node_i, comm_nodes])
        ki = degrees[node_i]
        sum_tot = np.sum(degrees[comm_nodes])
        
        # Simplified Delta Q formula
        return (ki_in / m) - (sum_tot * ki) / (2 * m**2)

    improved = True
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        for i in nodes:
            old_comm = refined_partition[i]
            # Only check communities that node i is actually connected to (saves time!)
            neighbor_nodes = np.where(A[i, :] > 0)[0]
            neighbor_comms = np.unique(refined_partition[neighbor_nodes])
            
            best_delta = 0
            best_move = old_comm
            
            for target_comm in neighbor_comms:
                if target_comm == old_comm: continue
                
                delta_q = get_delta_q(i, target_comm, refined_partition)
                if delta_q > best_delta:
                    best_delta = delta_q
                    best_move = target_comm
                    improved = True
            
            refined_partition[i] = best_move
            
    return refined_partition

D-Wave LeapHybrid

In [ ]:
hybrid_stats = pd.DataFrame(columns=['threshold', 'partition', 'runtime', 'num_communities', 'modularity', 'conductance']) # no community_sizes for hybrid since we will optimize after sampling

sampler = LeapHybridSampler()

for i in range(5, 20):
    G_undirected = graphs[i]['G_undirected']
    A = graphs[i]['A']
    N = graphs[i]['N']

    # Estimate reasonable number of communities
    num_communities_target = max(2, int(np.sqrt(N)))
    print(f"Target number of communities: {num_communities_target}")

    # Build QUBO
    print(f"\nBuilding QUBO matrix...")
    Q_dict, k_communities = build_modularity_qubo(A, num_communities_target, verbose=True)
    print(f"QUBO built with {len(Q_dict)-1} terms")

    # Convert QUBO to BQM (Binary Quadratic Model) for D-Wave
    Q_dict_clean = {}
    const_offset = 0

    for k, v in Q_dict.items():
        if k == '__const__':
            # Extract constant term for later addition
            const_offset = float(v)
        elif isinstance(k, tuple):
            Q_dict_clean[k] = float(v)
        else:
            node_idx = int(k)
            Q_dict_clean[(node_idx, node_idx)] = float(v)

    # Create BQM and add the constant offset
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict_clean)

    # Add constant term to the BQM's offset
    bqm.offset += const_offset

    print(f"BQM created: {bqm.num_variables} variables, {len(bqm.quadratic)} interactions")

    # Hybrid sampling with D-Wave
    start_time = time.time()
    sampleset_hybrid = sampler.sample(bqm, time_limit=10)

    hybrid_runtime = time.time() - start_time

    best_sample_hybrid = sampleset_hybrid.first.sample
    best_energy_hybrid = sampleset_hybrid.first.energy

    # Refine with greedy optimization
    partition_hybrid = extract_communities_from_sample(best_sample_hybrid, N, k_communities)
    print("Applying greedy refinement to hybrid solution...")
    partition_hybrid_optimized = greedy_refinement(A, partition_hybrid)

    num_communities_found_hybrid = len(np.unique(partition_hybrid))
    conductance_hybrid = compute_conductance(partition_hybrid, A)
    mod_raw = compute_modularity(partition_hybrid, A)
    mod_opt = compute_modularity(partition_hybrid_optimized, A)

    
    hybrid_stats.loc[len(hybrid_stats)] = {
        'threshold': i*0.01,
        'partition': partition_hybrid,
        'runtime': hybrid_runtime,
        'num_communities': num_communities_found_hybrid,
        'modularity': mod_opt,
        'conductance': conductance_hybrid,
    }

    print(f"\nGraph {i-4} results:")
    print(f"Runtime: {hybrid_runtime:.4f}s")
    print(f"Best energy: {best_energy_hybrid:.6f}")
    print(f"Number of communities found: {num_communities_found_hybrid}")
    print(f"Original Hybrid Modularity: {mod_raw:.6f}")
    print(f"Optimized Hybrid Modularity: {mod_opt:.6f}")
    print(f"Conductance: {conductance_hybrid:.6f}")
    print(f"Community sizes: {sorted(np.bincount(partition_hybrid), reverse=True)}")

hybrid_stats.to_csv('stats_by_threshold/hybrid_stats.csv', index=False)

Target number of communities: 11

Building QUBO matrix...
Modularity scale (max |coeff|): 0.004049
Penalty weight (lambda): 5.000000
Constant term (N*lambda): 620.00
Total QUBO terms: 92070 + 1 const
QUBO built with 92070 terms
BQM created: 1364 variables, 90706 interactions
Applying greedy refinement to hybrid solution...
Starting refinement... Initial Modularity: 0.098123
Refinement complete. Optimized Modularity: 0.727675

Graph 1 results:
Runtime: 3.9144s
Best energy: -0.043823
Number of communities found: 11
Original Hybrid Modularity: 0.098123
Optimized Hybrid Modularity: 0.727675
Conductance: 0.808586
Community sizes: [np.int64(17), np.int64(13), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(10), np.int64(6), np.int64(4)]
Target number of communities: 10

Building QUBO matrix...
Modularity scale (max |coeff|): 0.005076
Penalty weight (lambda): 5.000000
Constant term (N*lambda): 570.00
Total QUBO terms: 70680 + 1 const
QUBO built wit